# VQE 教程：顶层接口 + 手动 Parameter-Shift 梯度下降

本教程分两部分：
1. 顶层：直接使用 `VQERunner.run_model` 跑 VQE
2. 拆解：手动实现 `parameter-shift` 梯度 + 梯度下降更新

默认用 `Simulator`，便于稳定复现与快速实验。

In [1]:
import numpy as np

from quantum_hw import QuantumHardwareClient, VQERunner
from quantum_hw.algorithms.vqe import (
    build_ising_hamiltonian,
    build_hardware_efficient_ansatz,
)

def section(title: str):
    print("\n" + "=" * 18, title, "=" * 18)

## 1) 问题设置（Ising 哈密顿量）

In [2]:
section("setup")
num_qubits = 2
layers = 1
shots = 2048
max_iters = 8
learning_rate = 0.2
shift = np.pi / 2
seed = 42

hamiltonian = build_ising_hamiltonian(num_qubits=num_qubits, j=1.0, h=1.0)
observables = [obs for _, obs in hamiltonian]

print("hamiltonian terms:")
for coeff, obs in hamiltonian:
    print(f"  {coeff:+.3f} * {obs}")


================== setup ==================
hamiltonian terms:
  -1.000 * Z0 Z1
  -1.000 * X0
  -1.000 * X1


## 2) 顶层调用：`VQERunner.run_model`

直接把优化细节交给框架。

In [3]:
section("high-level VQE")
client = QuantumHardwareClient()
vqe = VQERunner(
    client=client,
    layers=layers,
    shots=shots,
    max_iters=max_iters,
    learning_rate=learning_rate,
    shift=shift,
    zne=False,
    readout_mitigation=False,
    seed=seed,
)

result_top = vqe.run_model(
    name="tutorial_vqe_top",
    num_qubits=num_qubits,
    model="ising",
    model_params={"j": 1.0, "h": 1.0},
    prefer_chips="Simulator",
)

print("best_energy:", result_top.best_energy)
print("best_params (first 6):", result_top.best_params[:6])
print("energy_history:", result_top.energy_history)


================== high-level VQE ==================
[vqe] prepare run: name=tutorial_vqe_top num_qubits=2 model=ising layers=1 shots=2048 max_iters=8
[vqe] candidate chips: ['Simulator']
[vqe] running on chip: Simulator
[vqe] start optimization: iters=8 layers=1 shots=2048 params=8 shift=1.5707963267948966
[vqe] iter 0 start
[vqe] iter 0 energy=0.762695 grad_norm=1.908319
[vqe] iter 0 new best energy=0.762695
[vqe] iter 1 start
[vqe] iter 1 energy=-0.129883 grad_norm=1.968281
[vqe] iter 1 new best energy=-0.129883
[vqe] iter 2 start
[vqe] iter 2 energy=-0.874023 grad_norm=1.165366
[vqe] iter 2 new best energy=-0.874023
[vqe] iter 3 start
[vqe] iter 3 energy=-1.068359 grad_norm=0.523425
[vqe] iter 3 new best energy=-1.068359
[vqe] iter 4 start
[vqe] iter 4 energy=-0.857422 grad_norm=1.195439
[vqe] iter 5 start
[vqe] iter 5 energy=-0.763672 grad_norm=1.507153
[vqe] iter 6 start
[vqe] iter 6 energy=-0.817383 grad_norm=1.583550
[vqe] iter 7 start
[vqe] iter 7 energy=-1.057617 grad_norm=1

## 3) 拆解版：手动写能量评估 + Parameter-Shift 梯度

思路：
- 给定参数 `params` 构造 ansatz 线路
- 用 `client.run_auto(... observables=...)` 得到每个 Pauli 项期望
- 按哈密顿量线性组合得到能量
- 用 parameter-shift 公式算梯度

In [4]:
section("manual energy + gradient")
client_manual = QuantumHardwareClient()

def evaluate_energy(params: np.ndarray, *, run_name: str):
    qc = build_hardware_efficient_ansatz(num_qubits, params, layers=layers)
    # Note: In a real application, you would want to reuse the same hardware for all evaluations to reduce tranpilation cost. Here we run separate tasks for simplicity.
    result = client_manual.run_auto(
        circuit=qc,
        name=run_name,
        num_qubits=num_qubits,
        shots=shots,
        observables=observables,
        prefer_chips="Simulator",
        print_true=False,
    )
    exp_map = result.observable_values
    energy = float(sum(coeff * exp_map.get(obs, 0.0) for coeff, obs in hamiltonian))
    return energy, exp_map

def parameter_shift_grad(params: np.ndarray, *, shift_value: float):
    grads = np.zeros_like(params, dtype=float)
    for i in range(params.size):
        p_plus = params.copy()
        p_minus = params.copy()
        p_plus[i] += shift_value
        p_minus[i] -= shift_value

        e_plus, _ = evaluate_energy(p_plus, run_name=f"tutorial_vqe_grad_p{i}")
        e_minus, _ = evaluate_energy(p_minus, run_name=f"tutorial_vqe_grad_m{i}")
        grads[i] = 0.5 * (e_plus - e_minus)
    return grads

num_params = 2 * num_qubits * (layers + 1)
rng = np.random.default_rng(seed)
params0 = rng.uniform(0.0, 2.0 * np.pi, size=num_params)

e0, exp0 = evaluate_energy(params0, run_name="tutorial_vqe_manual_init")
g0 = parameter_shift_grad(params0, shift_value=shift)

print("initial energy:", e0)
print("initial grad norm:", float(np.linalg.norm(g0)))
print("initial expectations:", exp0)


================== manual energy + gradient ==================
initial energy: 0.7587890625
initial grad norm: 1.8951399936147064
initial expectations: {'Z0 Z1': -0.310546875, 'X0': -0.2099609375, 'X1': -0.23828125}


## 4) 手动梯度下降循环

这里用最朴素的 GD 更新：
$$\theta \leftarrow \theta - \eta \nabla E(\theta)$$

In [ ]:
section("manual gradient descent")
manual_iters = 6
params = params0.copy()
energy_history_manual = []

for it in range(manual_iters):
    energy, _ = evaluate_energy(params, run_name=f"tutorial_vqe_manual_iter{it}")
    grads = parameter_shift_grad(params, shift_value=shift)
    grad_norm = float(np.linalg.norm(grads))

    energy_history_manual.append(float(energy))
    print(f"iter={it:02d} energy={energy:.6f} grad_norm={grad_norm:.6f}")

    params = params - learning_rate * grads

energy_final, exp_final = evaluate_energy(params, run_name="tutorial_vqe_manual_final")
print("manual final energy:", energy_final)
print("manual final expectations:", exp_final)


================== manual gradient descent ==================
iter=00 energy=0.812500 grad_norm=1.901052


## 5) 顶层与手动版本对比

In [ ]:
section("compare")
print("top best energy:", result_top.best_energy)
print("manual final energy:", energy_final)
print("top history:", result_top.energy_history)
print("manual history:", energy_history_manual)


================== compare ==================
top best energy: -1.0673828125
manual final energy: -1.5341796875
top history: [0.8671875, 0.0517578125, -0.490234375, -0.923828125, -1.0673828125, -0.9833984375, -0.8193359375, -0.7919921875]
manual history: [0.7421875, 0.255859375, -0.4091796875, -0.98828125, -1.3076171875, -1.482421875]


## 6) 下一步建议

- 把 `num_qubits` 增加到 4 或 6，观察参数量和收敛速度。
- 把手动 GD 改成 Adam（对应库内默认优化器）。